In [15]:
import requests
import torch
import re
import time
import psutil
import subprocess
from unidecode import unidecode
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, hamming_loss

import pandas as pd

from datasets import load_dataset

In [16]:
ds = load_dataset("TimSchopf/arxiv_categories", "default")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20397 entries, 0 to 20396
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   id             20397 non-null  object             
 1   title          20397 non-null  object             
 2   abstract       20397 non-null  object             
 3   categories     20397 non-null  object             
 4   creation_date  20397 non-null  datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), object(4)
memory usage: 796.9+ KB


In [17]:
test = test.rename(columns={'title': 'text'})
test = test.rename(columns={'categories': 'labels'})

In [18]:
def clean_element(lst):
    final = []
    for elem in lst:
        clean = elem.split('->')[-1]
        final.append(clean)
    return final

test['labels'] = test['labels'].apply(clean_element)

test

,id,text,abstract,labels,creation_date
0,2208.11718,gSwin: Gated MLP Vision Model with Hierarchica...,"Following the success in language domain, the ...","[cs.CV, cs.LG]",2022-08-24 18:00:46+00:00
1,2302.13571,FLAG: Fast Label-Adaptive Aggregation for Mult...,Federated learning aims to share private data ...,"[cs.AI, cs.LG]",2023-02-27 08:16:39+00:00
2,2208.12842,Advances in device-independent quantum key dis...,Device-independent quantum key distribution (D...,[quant-ph],2022-08-26 18:55:40+00:00
3,2303.07834,Finite-Horizon Constrained MDPs With Both Addi...,This paper considers the problem of finding a ...,"[math.OC, math.PR]",2023-03-14 12:15:58+00:00
4,1504.07490,Exact equations for structure functions and eq...,We derive equations for the source terms appea...,[physics.flu-dyn],2015-04-28 14:17:44+00:00
...,...,...,...,...,...
20392,1608.05170,Resource efficient redundancy using quorum-bas...,In this paper we propose a cycle redundancy te...,[cs.NI],2016-08-18 04:35:50+00:00
20393,cond-mat/9504003,An almost sure large deviation principle for t...,We prove a large deviation principle for the f...,[cond-mat],1995-04-03 11:11:22+00:00
20394,1412.5555,Local stability of Kolmogorov forward equation...,The focus of this work is on local stability o...,[math.PR],2014-12-17 20:03:13+00:00
20395,astro-ph/0408428,An Analysis of the Large Scale N-body Simulati...,We analyze the Minkowski functionals with a la...,[astro-ph],2004-08-24 06:12:43+00:00


In [19]:
labels = ["cs.AI", "cs.CL", "stat.ML", "math.OC", "cs.LG"]

test = test[test['labels'].apply(lambda cats: all(c in labels for c in cats))]
test = test[test['labels'].apply(len) > 0]
test.drop(columns=['id','abstract','creation_date'], inplace=True)
test.reset_index(drop=True, inplace=True)

test

,text,labels
0,FLAG: Fast Label-Adaptive Aggregation for Mult...,"[cs.AI, cs.LG]"
1,Large-Margin Classification in Hyperbolic Space,"[cs.LG, stat.ML]"
2,Student Specialization in Deep ReLU Networks W...,"[cs.LG, stat.ML]"
3,Joint Modelling of Emotion and Abusive Languag...,"[cs.CL, cs.LG]"
4,Allocation of Excitation Signals for Generic I...,[math.OC]
...,...,...
1265,Benign Overfitting in Two-layer Convolutional ...,"[cs.LG, math.OC, stat.ML]"
1266,An FPGA-Based On-Device Reinforcement Learning...,"[cs.LG, stat.ML]"
1267,Something for (almost) nothing: Improving deep...,[cs.LG]
1268,An Alternative to Variance: Gini Deviation for...,"[cs.AI, cs.LG]"


In [20]:
mlb = MultiLabelBinarizer()
test_labels_binarized = mlb.fit_transform(test['labels'])

test_labels_df = pd.DataFrame(test_labels_binarized, columns=mlb.classes_)

test = pd.concat([test, test_labels_df], axis=1)

test.drop(columns=['labels'], inplace=True)

test

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML
0,FLAG: Fast Label-Adaptive Aggregation for Mult...,1,0,1,0,0
1,Large-Margin Classification in Hyperbolic Space,0,0,1,0,1
2,Student Specialization in Deep ReLU Networks W...,0,0,1,0,1
3,Joint Modelling of Emotion and Abusive Languag...,0,1,1,0,0
4,Allocation of Excitation Signals for Generic I...,0,0,0,1,0
...,...,...,...,...,...,...
1265,Benign Overfitting in Two-layer Convolutional ...,0,0,1,1,1
1266,An FPGA-Based On-Device Reinforcement Learning...,0,0,1,0,1
1267,Something for (almost) nothing: Improving deep...,0,0,1,0,0
1268,An Alternative to Variance: Gini Deviation for...,1,0,1,0,0


In [21]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_18772\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


95887360

In [22]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [23]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multilabel classification tasks based on user instructions."},
        {"role": "user", "content": f"Classify the following text based on the task: Classification of computer science-based arXiv articles. Only respond with the labels that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
    ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}

    list = []

    if 'artificial intelligence' in classification_text:
        list.append('cs.AI')
    if 'computation and language' in classification_text:
        list.append('cs.CL')
    if 'statistics machine learning' in classification_text:
        list.append('stat.ML')
    if 'optimization and control' in classification_text:
        list.append('math.OC')
    if 'computational machine learning' in classification_text:
        list.append('cs.LG')
    
    print(f"Text: {text}")
    print(f"Response: {list}")
    
    return list, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [24]:
labels_for_prompt = ["Artificial Intelligence", "Computation and Language", "Statistics Machine Learning", "Optimization and Control", "Computational Machine Learning"]

# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels_for_prompt)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_18772\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-15T18:54:47.4704205Z', 'message': {'role': 'assistant', 'content': 'The text discusses a method for multi-label classification within a federated learning environment, focusing on efficient label-adaptive aggregation techniques. The most fitting classification is:\n\n**Machine Learning**', 'thinking': 'Okay, so I need to classify this text into one of the given categories. Let me read it carefully first.\n\nThe text is "FLAG: Fast Label-Adaptive Aggregation for Multi-label Classification in Federated Learning." Hmm, that\'s a bit confusing at first glance. The word "FLAG" stands out to me because it sounds like an acronym or maybe a specific term related to the topic. \n\nNow, looking at the possible labels: Artificial Intelligence, Computation and Language, Statistics Machi

In [25]:
for label in labels:
    test[f"{label} pred"] = test.apply(lambda row: 1 if label in row['prediction'] else 0, axis=1)

test = test.drop(columns=['prediction'])

test.to_csv('results/deepseekR1_ZS_multilabel3.csv', index=False)
test

,text,cs.AI,cs.CL,cs.LG,math.OC,stat.ML,label_counts,response_time,vram_usage,ram_usage,total_time,response_text,cs.AI pred,cs.CL pred,stat.ML pred,math.OC pred,cs.LG pred
0,FLAG: Fast Label-Adaptive Aggregation for Mult...,1,0,1,0,0,"{'Artificial Intelligence': 0, 'Computation an...",4.925799,2320,89.898438,2.882283,"Okay, so I need to classify this text into one...",0,0,0,0,0
1,Large-Margin Classification in Hyperbolic Space,0,0,1,0,1,"{'Artificial Intelligence': 0, 'Computation an...",4.417967,2364,90.093750,2.366167,"Okay, so I need to classify this text into one...",0,0,0,0,1
2,Student Specialization in Deep ReLU Networks W...,0,0,1,0,1,"{'Artificial Intelligence': 1, 'Computation an...",5.550508,2292,90.789062,3.492158,"Okay, so I need to classify this text into one...",1,0,0,0,0
3,Joint Modelling of Emotion and Abusive Languag...,0,1,1,0,0,"{'Artificial Intelligence': 1, 'Computation an...",3.712592,2327,90.902344,1.671202,"Okay, so I need to classify this text into one...",1,0,0,0,0
4,Allocation of Excitation Signals for Generic I...,0,0,0,1,0,"{'Artificial Intelligence': 0, 'Computation an...",3.894670,2374,90.535156,1.843146,"Alright, let's break this down step by step. F...",0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1265,Benign Overfitting in Two-layer Convolutional ...,0,0,1,1,1,"{'Artificial Intelligence': 0, 'Computation an...",5.714421,2329,76.484375,3.655658,"Alright, let's break this down step by step. F...",0,0,1,0,0
1266,An FPGA-Based On-Device Reinforcement Learning...,0,0,1,0,1,"{'Artificial Intelligence': 0, 'Computation an...",5.985725,2329,77.289062,3.926180,"Alright, let's break this down step by step. I...",0,0,0,0,1
1267,Something for (almost) nothing: Improving deep...,0,0,1,0,0,"{'Artificial Intelligence': 0, 'Computation an...",5.050612,2336,77.488281,3.006540,"Okay, so I need to classify this text into one...",0,0,0,1,0
1268,An Alternative to Variance: Gini Deviation for...,1,0,1,0,0,"{'Artificial Intelligence': 0, 'Computation an...",5.577207,2329,76.425781,3.544033,"Okay, so I need to classify this text into one...",0,0,1,1,1


In [26]:
y_true = test[labels].values
y_pred = test[[f"{label} pred" for label in labels]].values

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)
hamming_loss = hamming_loss(y_true, y_pred)
print('Hamming loss: %f' % hamming_loss)

Accuracy: 0.183465
F1 score: 0.486805
Precision: 0.543356
Recall: 0.478013
Hamming loss: 0.310866


In [27]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 5.017079787742435
Average VRAM usage: 2305.4543307086615
Average RAM usage: 77.10903358759843
Average total time: 2.9714149325984254


In [28]:
# save results to txt
with open('results/deepseekR1_ZS_multilabel3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')